# Week 04 Lab — Counting, N-grams, Distributional Semantics, and HMMs

## Learning goals
- Implement word counting and inspect frequency effects.
- Train a **bigram language model**, observe **data sparsity**, and apply **add-one smoothing**.
- Compute **sentence probability** and **perplexity**.
- Build **co-occurrence vectors** and compute **cosine similarity** for word similarity.
- Implement **HMM** parameter estimation + **forward** and **Viterbi** algorithms.


## Part 0 — Setup
Run these cells to install packages and import libraries.

In [ ]:
!pip install -q numpy nltk

In [ ]:
import math
import re
import random
from collections import Counter, defaultdict

import numpy as np
import nltk

nltk.download("punkt")
random.seed(0)
np.random.seed(0)

## Part 1 — Word counting

Tokenization decisions matter. We'll implement a minimal tokenizer and count words.

In [ ]:
toy_text = """
NLP is fun. NLP is useful!
Is NLP only about words? Not only words.
""".strip()

print(toy_text)

In [ ]:
PUNCT_RE = re.compile(r"^\W+$")

def tokenize(text: str) -> list[str]:
    """Lowercase, whitespace split, drop pure-punctuation tokens."""
    # TODO: implement
    pass

def word_counts(tokens: list[str]) -> dict[str, int]:
    # TODO: implement
    pass

tokens = tokenize(toy_text)
counts = word_counts(tokens)
len(tokens), list(counts.items())[:10]

In [ ]:
def top_k(counts: dict[str, int], k: int = 10):
    # TODO: implement
    pass

top_k(counts, k=10)

### Questions
1. What tokens dominate the top list? Are they content words or function words?
2. If you kept punctuation tokens, how would the top list change?

## Part 2 — Bigram language model + data sparsity

We'll build a bigram LM from a small corpus of sentences.

In [ ]:
sentences = [
    "i like nlp",
    "i like deep learning",
    "i like learning",
    "nlp is fun",
    "deep learning is fun",
]

sents_tok = [nltk.word_tokenize(s.lower()) for s in sentences]
sents_tok

In [ ]:
BOS = "<s>"
EOS = "</s>"
UNK = "<UNK>"

def add_boundaries(tokens: list[str]) -> list[str]:
    return [BOS] + tokens + [EOS]

sents_b = [add_boundaries(t) for t in sents_tok]

unigram_counts = Counter()
bigram_counts = Counter()
vocab = set([BOS, EOS, UNK])

# TODO: build unigram_counts, bigram_counts, and vocab

V = len(vocab)
V, list(sorted(vocab))[:15]

In [ ]:
def normalize_token(w: str, vocab: set[str]) -> str:
    # TODO: implement
    pass

def p_bigram_mle(w_prev: str, w: str) -> float:
    """Unsmoothed MLE bigram probability."""
    # TODO: implement
    pass

def p_bigram_add1(w_prev: str, w: str) -> float:
    """Add-one (Laplace) smoothed bigram probability."""
    # TODO: implement
    pass

print("Example MLE P(like|i) =", p_bigram_mle("i", "like"))
print("Example add1 P(like|i) =", p_bigram_add1("i", "like"))
print("Example add1 P(dragon|i) =", p_bigram_add1("i", "dragon"))

In [ ]:
# Data sparsity check
observed_bigram_types = sum(1 for _, c in bigram_counts.items() if c > 0)
possible_bigrams = V * V
sparsity_ratio = observed_bigram_types / possible_bigrams if possible_bigrams else 0.0

observed_bigram_types, possible_bigrams, sparsity_ratio

### Questions
1. Why does sparsity get worse as vocabulary grows?
2. Why does an unsmoothed LM assign probability 0 to many sentences?

## Part 3 — Sentence probability and perplexity

Compute log-probability and perplexity under the bigram model.

In [ ]:
def sentence_logprob(tokens: list[str], p_bigram) -> float:
    """Return sum of log probs for bigrams in a sentence with boundaries."""
    # TODO: implement
    pass

def perplexity(sentences_tokens: list[list[str]], p_bigram) -> float:
    """Perplexity over a list of tokenized sentences (each without boundaries)."""
    # TODO: implement
    pass

test_sents = [
    nltk.word_tokenize("i like nlp"),
    nltk.word_tokenize("nlp is fun"),
    nltk.word_tokenize("i like dragons"),
]

print("PP (MLE):", perplexity(test_sents, p_bigram_mle))
print("PP (add1):", perplexity(test_sents, p_bigram_add1))

### Questions
1. Which model has lower perplexity on sentences with unseen words, and why?
2. Why do we compute in log-space?

## Part 4 — Co-occurrence vectors

Build a word-context co-occurrence matrix with a fixed window size.

In [ ]:
def build_cooccurrence(sentences_tokens: list[list[str]], window: int = 2, min_count: int = 1):
    """Return X (word x context), word_to_id, id_to_word."""
    # TODO: implement
    pass

X, word_to_id, id_to_word = build_cooccurrence(sents_tok, window=2, min_count=1)
X.shape, list(word_to_id.items())[:10]

In [ ]:
def vector_for(word: str, X: np.ndarray, word_to_id: dict[str, int]) -> np.ndarray:
    # TODO: implement
    pass

vector_for("nlp", X, word_to_id)

## Part 5 — Cosine similarity for word similarity

Use cosine similarity over co-occurrence vectors.

In [ ]:
def cosine(u: np.ndarray, v: np.ndarray) -> float:
    # TODO: implement
    pass

def most_similar(query_word: str, X: np.ndarray, word_to_id: dict[str, int], id_to_word: dict[int, str], top_k: int = 5):
    # TODO: implement
    pass

most_similar("nlp", X, word_to_id, id_to_word, top_k=5)

### Questions
1. Are the nearest neighbors synonyms or topic-related words?
2. Give one failure case and explain why it happens.

## Part 6 — Hidden Markov Models (HMMs)

We'll use a tiny labeled dataset (states = tags, observations = words) to estimate parameters and decode with Viterbi.

In [ ]:
# Tiny labeled sequences: (state, word)
# Example: POS-like tags for demonstration only.
labeled_sequences = [
    [("PRON", "i"), ("VERB", "like"), ("NOUN", "nlp")],
    [("PRON", "i"), ("VERB", "like"), ("NOUN", "learning")],
    [("NOUN", "nlp"), ("VERB", "is"), ("ADJ", "fun")],
    [("NOUN", "learning"), ("VERB", "is"), ("ADJ", "fun")],
]

states = sorted({st for seq in labeled_sequences for (st, _) in seq})
vocab_hmm = sorted({w for seq in labeled_sequences for (_, w) in seq} | {UNK})
states, vocab_hmm

In [ ]:
def estimate_hmm_params(labeled_sequences, states, vocab, add_k: float = 1.0):
    """Estimate pi, A, B with optional add-k smoothing."""
    # Returns:
    # - pi: dict[state] -> prob
    # - A: dict[(prev_state, state)] -> prob
    # - B: dict[(state, word)] -> prob
    # TODO: implement
    pass

pi, A, B = estimate_hmm_params(labeled_sequences, states, vocab_hmm, add_k=1.0)
list(pi.items()), list(A.items())[:5], list(B.items())[:5]

In [ ]:
def hmm_forward(obs: list[str], states, pi, A, B, unk: str = UNK) -> float:
    """Forward algorithm: return P(obs)."""
    # TODO: implement
    pass

def hmm_viterbi(obs: list[str], states, pi, A, B, unk: str = UNK) -> list[str]:
    """Viterbi decoding: return most likely state sequence."""
    # TODO: implement
    pass

obs = ["i", "like", "fun"]
print("P(obs) =", hmm_forward(obs, states, pi, A, B))
print("best states =", hmm_viterbi(obs, states, pi, A, B))

### Questions
1. Why does forward use sums while Viterbi uses max?
2. What independence assumptions does an HMM make?


## Submission
- Ensure all `TODO` sections are completed.
- Answer the written questions in markdown cells.
